In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import tchc

In [ ]:
tned = r"T:\STORAGE-63T\2025RP_final\2050_v2"
hwynet = gpd.read_file(os.path.join(tned,"input","EMMEOutputs.gdb"), layer='TNED_HwyNet')

In [ ]:
hwycovid = 3755
hwylink = hwynet[hwynet['HWYCOV0_ID']==hwycovid].squeeze()

In [ ]:
widths = [5]*26  # adjust based on file format
stations = pd.read_fwf(r"T:\projects\sr14\version14_3_0\network_build\data\sta.hrpct", widths=widths)
stations['AM_max'] = stations[['a%6', 'a%7', 'a%8']].max(axis=1)
stations['OP_max'] = stations[['o%0', 'o%1', 'o%2', 'o%3', 'o%4', 
                               'o%5', 'o%9', 'o%10', 'o%11', 'o%12', 
                               'o%13', 'o%14', 'o%18', 'o%19', 'o%20', 
                               'o%21', 'o%22', 'o%23']].max(axis=1)
stations['PM_max'] = stations[['p%15', 'p%16', 'p%17']].max(axis=1)

station_peak_period = [
    [[0]*5406,[0]*5406],
    [[0]*5406,[0]*5406],
    [[0]*5406,[0]*5406]
    ]

for row in stations[['sta', 'dir', 'AM_max', 'OP_max', 'PM_max']].itertuples():
    station_peak_period[0][row.dir-1][row.sta]=1/(row.AM_max/100)
    station_peak_period[1][row.dir-1][row.sta]=1/(row.OP_max/100)
    station_peak_period[2][row.dir-1][row.sta]=1/(row.PM_max/100)

In [ ]:
widths = [5]*10
gc = pd.read_fwf(r"T:\projects\sr14\version14_3_0\network_build\data\gc", widths=widths, header=None)
gc = gc.drop(columns=[0], axis=1)

signal_gc_lookup = []
for i in range(0,36,9):
    signal_gc_lookup.append(gc.iloc[i:i+9].values.tolist())

In [ ]:
parametersByYears = pd.read_csv(r"T:\STORAGE-63T\2025RP_final\2050_v2\input\parametersByYears.csv")
parameters_2050 = parametersByYears[parametersByYears['year']=='2050'].T.squeeze()

In [ ]:
link = tchc.TCHCLink(
    link_identifier=hwylink['HWYCOV0_ID'],
    link_name=hwylink['NM'],
    length_feet=hwylink['SHAPE_Length'],
    functional_class=hwylink['FC'],
    high_occupancy_vehicle_class=hwylink['HOV'],
    jurisdiction=hwylink['COJUR'],
    median_type=hwylink['MED'],
    directionality=hwylink['DIR'],
    traffic_count_identifier=0, # unsure what this might be
    station_identifier=hwylink['COSTAT'],
    project_identifier=hwylink['PROJ'],
    
    from_node_identifier=hwylink['AN'],
    to_node_identifier=hwylink['BN'],
    
    lane_count_by_period_and_direction=[[hwylink['ABLNA'], hwylink['BALNA']],[hwylink['ABLNMD'], hwylink['BALNMD']], [hwylink['ABLNP'], hwylink['BALNP']]],
    auxiliary_lane_count_by_direction=[hwylink['ABAU'],hwylink['BAAU']],

    planned_lane_capacity_by_direction=[hwylink['ABPLC'],hwylink['BAPLC']],

    cross_street_functional_class_by_direction=[hwynet[(hwynet['AN']==hwylink['BN'])|(hwynet['BN']==hwylink['BN'])]['FC'].max(), 
                                               hwynet[(hwynet['AN']==hwylink['AN'])|(hwynet['BN']==hwylink['AN'])]['FC'].max()],
    control_type_by_direction=[hwylink['ABCNT'],hwylink['BACNT']],
    through_lane_count_by_direction=[hwylink['ABTL'], hwylink['BATL']],
    right_turn_lane_count_by_direction=[hwylink['ABRL'], hwylink['BARL']],
    left_turn_lane_count_by_direction=[hwylink['ABLL'], hwylink['BALL']],
    green_cycle_value_by_direction=[hwylink['ABGC'], hwylink['BAGC']],

    toll_cost_by_period=[hwylink['TOLLA'], hwylink['TOLLMD'], hwylink['TOLLP']],

    external_zone_delay_cost= 0.0 # unsure what this might be
)

In [ ]:
context = tchc.TCHCContext(
    auto_operating_cost_per_mile=parameters_2050['aoc.fuel'],
    managed_lane_capacity_rate=1.0,
    freeway_capacity_rate=1.0,
    analysis_year=2050,
    approach_count={hwylink['BN']:len(hwynet[((hwynet['AN']==hwylink['BN'])|(hwynet['BN']==hwylink['BN']))&(hwynet['HWYCOV0_ID']!=hwycovid)]),
                    hwylink['AN']:len(hwynet[((hwynet['AN']==hwylink['AN'])|(hwynet['BN']==hwylink['AN']))&(hwynet['HWYCOV0_ID']!=hwycovid)])},
    ramp_meter_direction_by_traffic_count_identifier={0:0}, # related to traffic_count_identifier
    station_peak_period_factor=station_peak_period,
    roadway_safety_adjustment_factor_by_jurisdiction={jur:1.0 for jur in range(1,20)}, # unsure what this is
    signal_green_cycle_lookup=signal_gc_lookup,
    four_way_stop_green_cycle_lookup=signal_gc_lookup[3],
    two_way_stop_green_cycle_lookup=signal_gc_lookup[1][5], # no need to figure out what the FC of the link is?
    border_delay_minutes_lookup=[[[0.0]]] # this isn't used
)

In [ ]:
tchc.apply_tchc(link, context)

In [ ]:
period_capacity_script = pd.DataFrame([capacity[0] for capacity in link.period_capacity_by_period_and_direction] + [capacity[1] for capacity in link.period_capacity_by_period_and_direction],
                                      index=['ABCPA','ABCPMD','ABCPP', 'BACPA', 'BACPMD', 'BACPP']).rename(columns={0:'script'})
period_capacity_hwynet = pd.DataFrame(hwylink[['ABCPA','ABCPMD','ABCPP', 'BACPA', 'BACPMD', 'BACPP']]).rename(columns={hwylink.name:'hwynet'})

intersection_capacity_script = pd.DataFrame([capacity[0] for capacity in link.intersection_capacity_by_period_and_direction] + [capacity[1] for capacity in link.intersection_capacity_by_period_and_direction],
                                            index=['ABCXA','ABCXMD','ABCXP', 'BACXA', 'BACXMD', 'BACXP']).rename(columns={0:'script'})
intersection_capacity_hwynet = pd.DataFrame(hwylink[['ABCXA','ABCXMD','ABCXP', 'BACXA', 'BACXMD', 'BACXP']]).rename(columns={hwylink.name:'hwynet'})

hourly_capacity_script = pd.DataFrame([capacity[0] for capacity in link.hourly_capacity_by_period_and_direction] + [capacity[1] for capacity in link.hourly_capacity_by_period_and_direction],
                                      index=['ABCHA','ABCHMD','ABCHP', 'BACHA', 'BACHMD', 'BACHP']).rename(columns={0:'script'})
hourly_capacity_hwynet = pd.DataFrame(hwylink[['ABCHA','ABCHMD','ABCHP', 'BACHA', 'BACHMD', 'BACHP']]).rename(columns={hwylink.name:'hwynet'})

In [ ]:
period_capacity = pd.concat([period_capacity_script,period_capacity_hwynet], axis=1)
intersection_capacity = pd.concat([intersection_capacity_script,intersection_capacity_hwynet], axis=1)
hourly_capacity = pd.concat([hourly_capacity_script,hourly_capacity_hwynet], axis=1)

In [ ]:
capacity = pd.concat([period_capacity, intersection_capacity, hourly_capacity])
capacity.index.name = 'attribute'
capacity['diff'] = capacity['script'] - capacity['hwynet']
capacity